# LeetCode 1722 — Minimize Hamming Distance After Swap Operations bằng Python

Link đề: https://leetcode.com/problems/minimize-hamming-distance-after-swap-operations/ · Độ khó: **Medium** · Chủ đề: Array, Depth-First Search, Union Find

Skeleton LeetCode:

```python
class Solution:
    def minimumHammingDistance(self, source: List[int], target: List[int],
                               allowedSwaps: List[List[int]]) -> int:
        ...
```

Lộ trình: **BFS trên mọi hoán vị đạt được** của `source` (vét cạn, `n ≤ 7`) → nhận ra trong mỗi **thành phần liên thông** của đồ thị chỉ số, có thể sắp xếp **tùy ý** → **DFS tìm thành phần + đếm đa tập** (`Counter`) → **Union-Find (DSU)** → DSU + **một từ điển `(gốc, giá trị)`** tiết kiệm bộ nhớ.

## 1. Mục tiêu học

1. Mô hình hóa các phép hoán đổi được phép thành **đồ thị vô hướng trên chỉ số**; hiểu rằng hoán đổi theo cạnh nhiều lần cho phép tạo **mọi hoán vị** trong một thành phần liên thông.
2. Quy bài toán về **ghép đa tập**: trong mỗi thành phần, số vị trí khớp tối đa = `|multiset(source) ∩ multiset(target)|`.
3. Cài **Union-Find** với nén đường (path compression) và hợp theo kích thước (union by size).
4. Tìm thành phần liên thông bằng **DFS lặp** (không đệ quy) để tránh tràn stack với `n = 10⁵`.
5. Kiểm chứng bằng **BFS trên không gian hoán vị** cho `n` nhỏ.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên `source` và `target` cùng độ dài `n`, và mảng `allowedSwaps` với `allowedSwaps[i] = [a_i, b_i]` cho phép **đổi chỗ** phần tử ở chỉ số `a_i` và `b_i` của `source`. Mỗi cặp có thể dùng **bao nhiêu lần tùy ý, theo thứ tự tùy ý**.

**Khoảng cách Hamming** giữa hai mảng cùng độ dài là số vị trí `i` mà `source[i] != target[i]`.

Trả về khoảng cách Hamming **nhỏ nhất** giữa `source` và `target` sau khi thực hiện các phép đổi chỗ trên `source`.

**Ví dụ 1:** `source = [1,2,3,4]`, `target = [2,1,4,5]`, `allowedSwaps = [[0,1],[2,3]]` → `1`. Đổi `0↔1`: `[2,1,3,4]`; đổi `2↔3`: `[2,1,4,3]` — chỉ khác ở chỉ số 3.

**Ví dụ 2:** `source = [1,2,3,4]`, `target = [1,3,2,4]`, `allowedSwaps = []` → `2`.

**Ví dụ 3:** `source = [5,1,2,4,3]`, `target = [1,5,4,2,3]`, `allowedSwaps = [[0,4],[4,2],[1,3],[1,4]]` → `0`.

**Ràng buộc:**
- `n == source.length == target.length`, `1 ≤ n ≤ 10⁵`
- `1 ≤ source[i], target[i] ≤ 10⁵`
- `0 ≤ allowedSwaps.length ≤ 10⁵`, `allowedSwaps[i] = [a_i, b_i]`, `0 ≤ a_i, b_i ≤ n − 1`, `a_i ≠ b_i`

In [1]:
import random
import time
from collections import Counter, defaultdict, deque
from typing import List

examples = [
    (([1, 2, 3, 4], [2, 1, 4, 5], [[0, 1], [2, 3]]), 1),
    (([1, 2, 3, 4], [1, 3, 2, 4], []), 2),
    (([5, 1, 2, 4, 3], [1, 5, 4, 2, 3], [[0, 4], [4, 2], [1, 3], [1, 4]]), 0),
]
for (src, tgt, sw), want in examples:
    print(f"source={src}, target={tgt}, allowedSwaps={sw} -> kỳ vọng {want}")

def hamming(a, b):
    return sum(x != y for x, y in zip(a, b))

source=[1, 2, 3, 4], target=[2, 1, 4, 5], allowedSwaps=[[0, 1], [2, 3]] -> kỳ vọng 1
source=[1, 2, 3, 4], target=[1, 3, 2, 4], allowedSwaps=[] -> kỳ vọng 2
source=[5, 1, 2, 4, 3], target=[1, 5, 4, 2, 3], allowedSwaps=[[0, 4], [4, 2], [1, 3], [1, 4]] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Khoảng cách Hamming

In [2]:
print(hamming([1, 2, 3, 4], [2, 1, 4, 5]), hamming([2, 1, 4, 3], [2, 1, 4, 5]))

4 1


### 3.2 Đồ thị bằng danh sách kề

Đỉnh là chỉ số `0..n−1`; mỗi cặp `[a, b]` là một cạnh vô hướng. `defaultdict(list)` hoặc `[[] for _ in range(n)]`.

In [3]:
n = 5
adj = [[] for _ in range(n)]
for a, b in [[0, 4], [4, 2], [1, 3], [1, 4]]:
    adj[a].append(b); adj[b].append(a)
print(adj)

[[4], [3, 4], [4], [1], [0, 2, 1]]


### 3.3 `Counter` và giao đa tập

`Counter(a) & Counter(b)` lấy **min** số lần xuất hiện của từng giá trị — đúng là "số cặp ghép được" giữa hai đa tập. `sum(c.values())` cho kích thước.

In [4]:
c1, c2 = Counter([1, 2, 2, 3]), Counter([2, 2, 2, 4])
print(c1 & c2, sum((c1 & c2).values()))

Counter({2: 2}) 2


### 3.4 Union-Find (Disjoint Set Union)

Mỗi tập có một **gốc**. `find(x)` đi theo `parent` tới gốc (nén đường: gán thẳng `parent` về gốc). `union(a, b)` nối gốc nhỏ vào gốc lớn. Gần như `O(1)` mỗi thao tác (hàm Ackermann ngược `α(n)`).

In [5]:
class DSU:
    def __init__(self, n):
        self.p = list(range(n))
        self.sz = [1] * n
    def find(self, x):
        root = x
        while self.p[root] != root:
            root = self.p[root]
        while self.p[x] != root:          # nén đường (lặp, không đệ quy)
            self.p[x], x = root, self.p[x]
        return root
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.sz[ra] < self.sz[rb]:
            ra, rb = rb, ra
        self.p[rb] = ra
        self.sz[ra] += self.sz[rb]
        return True

d = DSU(5)
for a, b in [[0, 4], [4, 2], [1, 3]]:
    d.union(a, b)
print([d.find(i) for i in range(5)])

[0, 1, 0, 1, 0]


### 3.5 Hoán vị sinh bởi các phép đổi chỗ kề cạnh

Định lý quen thuộc: với một đồ thị liên thông trên tập chỉ số, các phép đổi chỗ theo cạnh (dùng nhiều lần) sinh ra **mọi hoán vị** của tập đó. Ví dụ đường đi `0 − 1 − 2`: chỉ cần đổi `(0,1)` và `(1,2)` là đạt cả 6 hoán vị của 3 phần tử (giống bubble sort chỉ dùng đổi chỗ kề).

In [6]:
def reachable_perms(arr, swaps):
    start = tuple(arr); seen = {start}; q = deque([start])
    while q:
        u = q.popleft()
        for a, b in swaps:
            v = list(u); v[a], v[b] = v[b], v[a]; v = tuple(v)
            if v not in seen:
                seen.add(v); q.append(v)
    return seen

print(len(reachable_perms([10, 20, 30], [[0, 1], [1, 2]])), "hoán vị đạt được (3! = 6)")
print(len(reachable_perms([10, 20, 30, 40], [[0, 1], [2, 3]])), "hoán vị đạt được (2!·2! = 4)")

6 hoán vị đạt được (3! = 6)
4 hoán vị đạt được (2!·2! = 4)


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 3: cạnh `0−4, 4−2, 1−3, 1−4` ⇒ mọi chỉ số `{0,1,2,3,4}` **liên thông** ⇒ có thể sắp `source` tùy ý. Đa tập `source = {5,1,2,4,3}` = đa tập `target = {1,5,4,2,3}` ⇒ khớp hết ⇒ Hamming `0`.

Ví dụ 1: hai thành phần `{0,1}` và `{2,3}`.

| thành phần | source tại đó | target tại đó | ghép được | không khớp |
|---|---|---|---|---|
| {0, 1} | {1, 2} | {2, 1} | 2 | 0 |
| {2, 3} | {3, 4} | {4, 5} | 1 (số 4) | 1 |

Tổng không khớp = `1`.

**Quy tắc:** đáp án = `n − Σ_{thành phần C} |Counter(source[C]) ∩ Counter(target[C])|`.

## 5. Cách 1 — Brute Force / baseline: BFS trên mọi trạng thái của `source`

**Ý tưởng:** BFS từ `source`, mỗi bước áp một phép đổi chỗ được phép; tính Hamming tối thiểu trên mọi trạng thái đạt được. Đúng theo định nghĩa (không cần định lý hoán vị).

In [7]:
def min_hamming_bfs(source, target, swaps):
    return min(hamming(s, target) for s in reachable_perms(source, swaps))

for (src, tgt, sw), want in examples:
    print(min_hamming_bfs(src, tgt, sw), "kỳ vọng", want)

1 kỳ vọng 1
2 kỳ vọng 2
0 kỳ vọng 0


**Độ phức tạp:** số trạng thái tới `n!` (khi liên thông và mọi giá trị khác nhau), mỗi trạng thái `O(m)` láng giềng và `O(n)` để tạo tuple ⇒ `O(n! · m · n)`. Chỉ dùng cho `n ≤ 8`.

**Nút thắt:** ta không cần biết **thứ tự** cụ thể — chỉ cần biết trong mỗi thành phần có thể sắp tùy ý ⇒ chuyển sang đếm.

In [8]:
for n in [5, 6, 7, 8]:
    src = list(range(n)); tgt = src[::-1]
    sw = [[i, i + 1] for i in range(n - 1)]
    t0 = time.perf_counter(); r = min_hamming_bfs(src, tgt, sw)
    print(f"n={n}: {r}, {time.perf_counter() - t0:.3f}s")

n=5: 0, 0.000s
n=6: 0, 0.001s
n=7: 0, 0.007s
n=8: 0, 0.068s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (thành phần = tự do hoán vị):** gọi `G` là đồ thị chỉ số với các cạnh `allowedSwaps`. Phần tử ở chỉ số `i` chỉ có thể di chuyển tới các chỉ số cùng thành phần với `i` (mỗi phép đổi giữ phần tử trong thành phần). Ngược lại, trong một thành phần liên thông, ta đạt được **mọi** hoán vị (mục 3.5). ⇒ Các thành phần **độc lập** với nhau.

**Insight 2 (ghép đa tập):** trong thành phần `C`, ta tự do đặt các giá trị `source[C]` vào các vị trí `C`. Số vị trí khớp tối đa = số cặp ghép được giữa đa tập `source[C]` và đa tập `target[C]` = `Σ_v min(cnt_s[v], cnt_t[v])`.

**Quy tắc thực thi:**
1. Gom chỉ số theo thành phần (DFS hoặc DSU).
2. Với mỗi thành phần: đếm `source`, rồi duyệt `target` — mỗi giá trị còn trong bộ đếm thì "ghép" (giảm 1), ngược lại tăng `ans`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS lặp tìm thành phần + `Counter`

In [9]:
def min_hamming_dfs(source, target, swaps):
    n = len(source)
    adj = [[] for _ in range(n)]
    for a, b in swaps:
        adj[a].append(b)
        adj[b].append(a)
    seen = [False] * n
    matched = 0
    for s in range(n):
        if seen[s]:
            continue
        comp = []
        stack = [s]
        seen[s] = True
        while stack:
            u = stack.pop()
            comp.append(u)
            for v in adj[u]:
                if not seen[v]:
                    seen[v] = True
                    stack.append(v)
        cs = Counter(source[i] for i in comp)
        ct = Counter(target[i] for i in comp)
        matched += sum((cs & ct).values())
    return n - matched

for (src, tgt, sw), want in examples:
    print(min_hamming_dfs(src, tgt, sw), "kỳ vọng", want)

1 kỳ vọng 1
2 kỳ vọng 2
0 kỳ vọng 0


**Độ phức tạp:** Time `O(n + m)` (DFS + đếm), Space `O(n + m)` cho danh sách kề.

### 7.2 Cách 3 — Union-Find + từ điển đếm theo gốc

Từng đoạn:
1. `union(a, b)` cho mọi cặp.
2. `cnt[find(i)][source[i]] += 1` cho mọi `i`.
3. Duyệt `i`: `r = find(i)`; nếu `cnt[r][target[i]] > 0` thì giảm (ghép được), ngược lại `ans += 1`.

In [10]:
def min_hamming_dsu(source, target, swaps):
    n = len(source)
    d = DSU(n)
    for a, b in swaps:
        d.union(a, b)
    cnt = defaultdict(Counter)
    for i, x in enumerate(source):
        cnt[d.find(i)][x] += 1
    ans = 0
    for i, y in enumerate(target):
        c = cnt[d.find(i)]
        if c[y] > 0:
            c[y] -= 1
        else:
            ans += 1
    return ans

for (src, tgt, sw), want in examples:
    print(min_hamming_dsu(src, tgt, sw), "kỳ vọng", want)

1 kỳ vọng 1
2 kỳ vọng 2
0 kỳ vọng 0


**Độ phức tạp:** Time `O((n + m) · α(n))`, Space `O(n)` (không cần lưu danh sách kề).

### 7.3 Cách 4 — DSU với mảng `parent` thuần + một dict khóa `(gốc, giá trị)`

Viết DSU bằng hàm cục bộ (nhanh hơn method) và một `dict` phẳng thay vì `defaultdict(Counter)` lồng nhau.

In [11]:
def min_hamming_flat(source, target, swaps):
    n = len(source)
    parent = list(range(n))

    def find(x):
        root = x
        while parent[root] != root:
            root = parent[root]
        while parent[x] != root:
            parent[x], x = root, parent[x]
        return root

    for a, b in swaps:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    roots = [find(i) for i in range(n)]
    cnt = {}
    for r, x in zip(roots, source):
        key = (r, x)
        cnt[key] = cnt.get(key, 0) + 1
    ans = 0
    for r, y in zip(roots, target):
        key = (r, y)
        c = cnt.get(key, 0)
        if c:
            cnt[key] = c - 1
        else:
            ans += 1
    return ans

for (src, tgt, sw), want in examples:
    print(min_hamming_flat(src, tgt, sw), "kỳ vọng", want)

1 kỳ vọng 1
2 kỳ vọng 2
0 kỳ vọng 0


Không dùng union by size ở đây; nhờ nén đường, độ phức tạp khấu hao vẫn `O(log n)` mỗi thao tác (đủ nhanh thực tế).

**Độ phức tạp:** Time `O((n + m) log n)` worst (thực tế gần tuyến tính), Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1:

In [12]:
def trace_dsu(source, target, swaps):
    n = len(source); d = DSU(n)
    for a, b in swaps:
        d.union(a, b)
        print(f"union({a}, {b}) -> gốc: {[d.find(i) for i in range(n)]}")
    cnt = defaultdict(Counter)
    for i, x in enumerate(source):
        cnt[d.find(i)][x] += 1
    print("đếm source theo gốc:", {r: dict(c) for r, c in cnt.items()})
    ans = 0
    for i, y in enumerate(target):
        r = d.find(i); c = cnt[r]
        if c[y] > 0:
            c[y] -= 1; res = "ghép được"
        else:
            ans += 1; res = "KHÔNG khớp"
        print(f"i={i}: target={y}, gốc={r}, {res:10s} -> còn lại {dict(c)}, ans={ans}")
    return ans

trace_dsu([1, 2, 3, 4], [2, 1, 4, 5], [[0, 1], [2, 3]])

union(0, 1) -> gốc: [0, 0, 2, 3]
union(2, 3) -> gốc: [0, 0, 2, 2]
đếm source theo gốc: {0: {1: 1, 2: 1}, 2: {3: 1, 4: 1}}
i=0: target=2, gốc=0, ghép được  -> còn lại {1: 1, 2: 0}, ans=0
i=1: target=1, gốc=0, ghép được  -> còn lại {1: 0, 2: 0}, ans=0
i=2: target=4, gốc=2, ghép được  -> còn lại {3: 1, 4: 0}, ans=0
i=3: target=5, gốc=2, KHÔNG khớp -> còn lại {3: 1, 4: 0}, ans=1


1

In [13]:
trace_dsu([5, 1, 2, 4, 3], [1, 5, 4, 2, 3], [[0, 4], [4, 2], [1, 3], [1, 4]])

union(0, 4) -> gốc: [0, 1, 2, 3, 0]
union(4, 2) -> gốc: [0, 1, 0, 3, 0]
union(1, 3) -> gốc: [0, 1, 0, 1, 0]
union(1, 4) -> gốc: [0, 0, 0, 0, 0]
đếm source theo gốc: {0: {5: 1, 1: 1, 2: 1, 4: 1, 3: 1}}
i=0: target=1, gốc=0, ghép được  -> còn lại {5: 1, 1: 0, 2: 1, 4: 1, 3: 1}, ans=0
i=1: target=5, gốc=0, ghép được  -> còn lại {5: 0, 1: 0, 2: 1, 4: 1, 3: 1}, ans=0
i=2: target=4, gốc=0, ghép được  -> còn lại {5: 0, 1: 0, 2: 1, 4: 0, 3: 1}, ans=0
i=3: target=2, gốc=0, ghép được  -> còn lại {5: 0, 1: 0, 2: 0, 4: 0, 3: 1}, ans=0
i=4: target=3, gốc=0, ghép được  -> còn lại {5: 0, 1: 0, 2: 0, 4: 0, 3: 0}, ans=0


0

Bảng tóm tắt ví dụ 3: sau 4 phép `union`, mọi chỉ số chung một gốc; bộ đếm `{5:1, 1:1, 2:1, 4:1, 3:1}`; duyệt `target = [1,5,4,2,3]` — mọi giá trị đều ghép được ⇒ `ans = 0`.

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def minimumHammingDistance(self, source: List[int], target: List[int],
                               allowedSwaps: List[List[int]]) -> int:
        n = len(source)
        parent = list(range(n))
        size = [1] * n

        def find(x: int) -> int:
            root = x
            while parent[root] != root:
                root = parent[root]
            while parent[x] != root:
                parent[x], x = root, parent[x]
            return root

        for a, b in allowedSwaps:
            ra, rb = find(a), find(b)
            if ra != rb:
                if size[ra] < size[rb]:
                    ra, rb = rb, ra
                parent[rb] = ra
                size[ra] += size[rb]

        cnt = defaultdict(Counter)
        for i, x in enumerate(source):
            cnt[find(i)][x] += 1
        ans = 0
        for i, y in enumerate(target):
            c = cnt[find(i)]
            if c[y] > 0:
                c[y] -= 1
            else:
                ans += 1
        return ans

sol = Solution()
for (src, tgt, sw), want in examples:
    got = sol.minimumHammingDistance(src, tgt, sw)
    print(got, "OK" if got == want else "SAI")

1 OK
2 OK
0 OK


## 10. Vì sao thuật toán đúng

**(a) Phần tử không rời thành phần.** Mỗi phép đổi chỗ `(a, b)` di chuyển hai phần tử giữa `a` và `b` — cùng thành phần. Quy nạp: mọi phần tử luôn ở trong thành phần của vị trí ban đầu. ⇒ Đa tập giá trị trên mỗi thành phần **bất biến**.

**(b) Mọi hoán vị trong thành phần đều đạt được.** Lấy cây khung của thành phần. Với cây, ta có thể đưa bất kỳ phần tử nào tới một **lá** bằng các phép đổi dọc đường đi, rồi "cố định" lá đó và xóa nó khỏi cây — lặp lại cho cây nhỏ hơn. Do đó đặt được giá trị mong muốn ở từng vị trí ⇒ mọi sắp xếp.

**(c) Tối ưu trong một thành phần.** Với đa tập `S` (giá trị của `source`) và các vị trí có giá trị đích `T`, số vị trí khớp không thể vượt `Σ_v min(S[v], T[v])` (mỗi giá trị `v` khớp nhiều nhất `min` lần). Và theo (b) đạt được đúng số đó (đặt các bản sao `v` vào các vị trí cần `v`, phần còn lại tùy ý).

**(d) Các thành phần độc lập** ⇒ cộng lại. Thuật toán "duyệt `target`, ghép tham lam nếu còn trong bộ đếm" tính đúng `Σ min` vì thứ tự ghép không ảnh hưởng tổng `min`.

## 11. Phân tích Time / Space

Gọi `n ≤ 10⁵` là độ dài, `m ≤ 10⁵` là số cặp swap.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. BFS hoán vị | `O(n! · m · n)` | `O(n! · n)` | `n ≤ 8` |
| 2. DFS + Counter | `O(n + m)` | `O(n + m)` | dựng danh sách kề |
| 3. DSU + Counter theo gốc | `O((n + m) α(n))` | `O(n)` | |
| 4. DSU phẳng + dict `(gốc, giá trị)` | `O((n + m) log n)` | `O(n)` | ít đối tượng Python |

In [15]:
random.seed(1)
n = 10**5
src = [random.randint(1, 10**5) for _ in range(n)]
tgt = [random.choice(src) if random.random() < 0.7 else random.randint(1, 10**5) for _ in range(n)]
sw = [[random.randrange(n), random.randrange(n)] for _ in range(10**5)]
sw = [[a, b] for a, b in sw if a != b]
for name, fn in [("dfs", min_hamming_dfs), ("dsu", min_hamming_dsu), ("flat", min_hamming_flat),
                 ("submit", Solution().minimumHammingDistance)]:
    t0 = time.perf_counter(); r = fn(src, tgt, sw)
    print(f"{name:6s}: {r}  {time.perf_counter() - t0:.3f}s")

dfs   : 52752  0.148s
dsu   : 52752  0.139s


flat  : 52752  0.166s
submit: 52752  0.138s


## 12. Bộ test

In [16]:
fns = [min_hamming_dfs, min_hamming_dsu, min_hamming_flat,
       lambda s, t, w: Solution().minimumHammingDistance(s, t, w)]

for (src, tgt, sw), want in examples:
    assert min_hamming_bfs(src, tgt, sw) == want
    for fn in fns:
        assert fn(src, tgt, sw) == want

edge = [
    (([1], [1], []), 0),
    (([1], [2], []), 1),
    (([1, 1, 2], [2, 1, 1], [[0, 2]]), 0),             # giá trị lặp
    (([1, 2, 3], [3, 2, 1], [[0, 1], [0, 1]]), 2),     # cặp trùng lặp
    (([1, 2, 3], [3, 1, 2], [[0, 1], [1, 2]]), 0),     # chuỗi liên thông
    (([7, 7, 7], [7, 8, 9], [[0, 1], [1, 2]]), 2),
]
for (src, tgt, sw), want in edge:
    assert min_hamming_bfs(src, tgt, sw) == want, (src, tgt, sw)
    for fn in fns:
        assert fn(src, tgt, sw) == want, (fn, src, tgt, sw)

random.seed(2024)
for _ in range(500):
    n = random.randint(1, 6)
    src = [random.randint(1, 3) for _ in range(n)]
    tgt = [random.randint(1, 3) for _ in range(n)]
    m = random.randint(0, 6) if n > 1 else 0
    sw = []
    for _ in range(m):
        a, b = random.sample(range(n), 2)
        sw.append([a, b])
    want = min_hamming_bfs(src, tgt, sw)
    for fn in fns:
        assert fn(src, tgt, sw) == want, (fn, src, tgt, sw)
print("ví dụ + edge + 500 bộ ngẫu nhiên khớp BFS hoán vị")

ví dụ + edge + 500 bộ ngẫu nhiên khớp BFS hoán vị


In [17]:
random.seed(7)
for _ in range(50):
    n = random.randint(1, 2000)
    src = [random.randint(1, 50) for _ in range(n)]
    tgt = [random.randint(1, 50) for _ in range(n)]
    sw = [random.sample(range(n), 2) for _ in range(random.randint(0, n))] if n > 1 else []
    r = min_hamming_dfs(src, tgt, sw)
    assert r == min_hamming_dsu(src, tgt, sw) == min_hamming_flat(src, tgt, sw) == Solution().minimumHammingDistance(src, tgt, sw)
print("n tới 2000: bốn cách nhanh khớp nhau")

n tới 2000: bốn cách nhanh khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ áp mỗi phép đổi **một lần**, theo thứ tự cho sẵn, khi có lợi

Tham lam cục bộ bỏ qua việc kết hợp nhiều phép (ví dụ đi vòng qua chỉ số trung gian).

In [18]:
def wrong_greedy_once(source, target, swaps):
    s = list(source)
    for a, b in swaps:
        before = (s[a] == target[a]) + (s[b] == target[b])
        after = (s[b] == target[a]) + (s[a] == target[b])
        if after > before:
            s[a], s[b] = s[b], s[a]
    return hamming(s, target)

for (src, tgt, sw), want in examples:
    print("sai =", wrong_greedy_once(src, tgt, sw), "| đúng =", want)

sai = 1 | đúng = 1
sai = 2 | đúng = 2
sai = 4 | đúng = 0


Ví dụ 3 cần đưa `5` từ chỉ số 0 sang chỉ số 1 qua đường `0 − 4 − 1` — không phép đổi đơn lẻ nào "có lợi ngay". **Sửa:** suy nghĩ theo thành phần liên thông.

### Lỗi 2 — Đếm theo **tập** (set) thay vì **đa tập**

Dùng `set(source[C]) & set(target[C])` bỏ qua giá trị lặp.

In [19]:
def wrong_set(source, target, swaps):
    n = len(source); d = DSU(n)
    for a, b in swaps:
        d.union(a, b)
    comps = defaultdict(list)
    for i in range(n):
        comps[d.find(i)].append(i)
    matched = 0
    for comp in comps.values():
        matched += len({source[i] for i in comp} & {target[i] for i in comp})   # SAI
    return n - matched

src, tgt, sw = [1, 1, 2], [1, 1, 3], [[0, 1], [1, 2]]
print("sai =", wrong_set(src, tgt, sw), "| đúng =", min_hamming_bfs(src, tgt, sw))

sai = 2 | đúng = 1


**Sửa:** `Counter` và lấy `min` số lần (`Counter & Counter`).

### Lỗi 3 — DSU không dùng `find` khi đếm (dùng `parent[i]` trực tiếp)

`parent[i]` chưa chắc là gốc nếu chưa nén đường ⇒ cùng thành phần nhưng khác "khóa".

In [20]:
def wrong_parent(source, target, swaps):
    n = len(source); parent = list(range(n))
    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x
    for a, b in swaps:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    cnt = defaultdict(Counter)
    for i, x in enumerate(source):
        cnt[parent[i]][x] += 1                  # SAI: phải là find(i)
    ans = 0
    for i, y in enumerate(target):
        c = cnt[parent[i]]
        if c[y] > 0: c[y] -= 1
        else: ans += 1
    return ans

src, tgt, sw = [5, 1, 2, 4, 3], [1, 5, 4, 2, 3], [[0, 4], [4, 2], [1, 3], [1, 4]]
print("sai =", wrong_parent(src, tgt, sw), "| đúng =", Solution().minimumHammingDistance(src, tgt, sw))

sai = 2 | đúng = 0


**Sửa:** luôn dùng `find(i)` để lấy gốc.

### Lỗi 4 — DFS đệ quy trên `n = 10⁵`

Thành phần là một đường dài `10⁵` đỉnh ⇒ đệ quy sâu `10⁵` vượt giới hạn mặc định. Minh họa an toàn bằng cách bắt ngoại lệ:

In [21]:
import sys

def comp_size_recursive(n, edges):
    adj = [[] for _ in range(n)]
    for a, b in edges:
        adj[a].append(b); adj[b].append(a)
    seen = [False] * n
    def dfs(u):
        seen[u] = True
        return 1 + sum(dfs(v) for v in adj[u] if not seen[v])
    return dfs(0)

n = 5000
edges = [[i, i + 1] for i in range(n - 1)]
try:
    print("kích thước:", comp_size_recursive(n, edges))
except RecursionError as e:
    print("RecursionError:", str(e)[:60], f"(giới hạn = {sys.getrecursionlimit()})")

RecursionError: maximum recursion depth exceeded (giới hạn = 1000)


**Sửa:** DFS bằng stack tường minh (Cách 2) hoặc DSU (Cách 3, 4).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Được đổi chỗ các cặp chỉ số **bao nhiêu lần tùy ý**" ⇒ **thành phần liên thông = tự do hoán vị** ⇒ DSU.
- Sau khi gom nhóm, bài toán thường thành **đếm/ghép đa tập** hoặc **sắp xếp trong nhóm** (ví dụ 1202 *Smallest String With Swaps*: sắp từng nhóm rồi đặt lại).
- DSU trong Python: viết `find` lặp + nén đường; tránh đệ quy.
- Kiểm chứng định lý "mọi hoán vị" bằng BFS trên `n` nhỏ trước khi tin.

## 15. Biến thể của bài

1. **LeetCode 1202 — Smallest String With Swaps:** cùng mô hình; trong mỗi thành phần sắp các ký tự tăng dần và đặt vào các chỉ số tăng dần.
2. **Trả về một cách sắp xếp `source` đạt Hamming tối thiểu:** ghép đa tập rồi gán phần dư tùy ý.
3. **Đếm số phép đổi tối thiểu** để đạt Hamming tối thiểu: khó hơn nhiều (liên quan chu trình hoán vị và khoảng cách trên đồ thị).
4. **Cạnh có trọng số / chi phí đổi chỗ:** không còn tự do — bài toán tối ưu khác.

In [22]:
def smallest_string_with_swaps(s, pairs):
    """Biến thể 1 (LeetCode 1202)."""
    n = len(s); d = DSU(n)
    for a, b in pairs:
        d.union(a, b)
    groups = defaultdict(list)
    for i in range(n):
        groups[d.find(i)].append(i)
    res = list(s)
    for idx in groups.values():
        chars = sorted(res[i] for i in idx)
        for i, ch in zip(sorted(idx), chars):
            res[i] = ch
    return "".join(res)

print(smallest_string_with_swaps("dcab", [[0, 3], [1, 2]]), smallest_string_with_swaps("dcab", [[0, 3], [1, 2], [0, 2]]))

bacd abcd


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS hoán vị | duyệt mọi trạng thái | `O(n!·m·n)` | lớn | kiểm chứng |
| 2. DFS + Counter | thành phần bằng DFS lặp | `O(n + m)` | `O(n + m)` | thích đồ thị tường minh |
| 3. DSU + Counter | gốc DSU làm khóa nhóm | `O((n+m)α)` | `O(n)` | **nộp bài** |
| 4. DSU phẳng | dict `(gốc, giá trị)` | `O((n+m) log n)` | `O(n)` | tối ưu hằng số |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `source = [1,2,3]`, `target = [2,3,1]`, `allowedSwaps = [[0,1]]`?

**Bài 2 (tính tay):** với `source = [4,4,1,2]`, `target = [1,4,4,3]`, `allowedSwaps = [[0,2],[1,3]]`: liệt kê thành phần và số ghép được.

**Bài 3 (điền TODO):** hoàn thiện `components(n, swaps)` trả về danh sách các thành phần (mỗi thành phần là list chỉ số đã sắp) bằng DSU.

**Bài 4 (tự viết lại):** viết `best_arrangement(source, target, swaps)` trả về **một mảng** đạt khoảng cách Hamming tối thiểu (là một hoán vị hợp lệ của `source` trong từng thành phần).

**Bài 5 (mini challenge):** đếm **số phép đổi chỗ tối thiểu** để biến `source` thành một sắp xếp đạt Hamming tối thiểu — giới hạn `n ≤ 7`, dùng BFS (biến thể tự đặt). In kết quả cho vài input.

In [23]:
def components(n, swaps):
    # TODO: DSU, gom chỉ số theo gốc, sắp mỗi nhóm và danh sách nhóm
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Thành phần `{0,1}` và `{2}`. (2) `{0,1}`: source `{1,2}`, target `{2,3}` ⇒ ghép 1. `{2}`: `3` vs `1` ⇒ 0. (3) Hamming `3 − 1 = 2`.

**Bài 2:** (1) Thành phần `{0,2}`, `{1,3}`. (2) `{0,2}`: `{4,1}` vs `{1,4}` ⇒ 2; `{1,3}`: `{4,2}` vs `{4,3}` ⇒ 1. (3) Hamming `4 − 3 = 1`.

**Bài 3:** (1) `d = DSU(n)`, union mọi cặp. (2) `groups[d.find(i)].append(i)`. (3) `sorted(sorted(g) for g in groups.values())`.

**Bài 4:** (1) Mỗi thành phần: `Counter` của source. (2) Vị trí nào target còn trong Counter thì đặt giá trị đó. (3) Các vị trí còn lại nhận phần dư bất kỳ.

**Bài 5:** (1) Tính Hamming tối thiểu `h*` (Cách 3). (2) BFS từ `source` theo các phép đổi, đếm tầng. (3) Dừng ở tầng đầu tiên có trạng thái với Hamming `= h*`.

## 19. Đáp án tham khảo

In [24]:
# Bài 1
assert Solution().minimumHammingDistance([1, 2, 3], [2, 3, 1], [[0, 1]]) == 2

# Bài 2
assert Solution().minimumHammingDistance([4, 4, 1, 2], [1, 4, 4, 3], [[0, 2], [1, 3]]) == 1

# Bài 3
def components_ans(n, swaps):
    d = DSU(n)
    for a, b in swaps:
        d.union(a, b)
    groups = defaultdict(list)
    for i in range(n):
        groups[d.find(i)].append(i)
    return sorted(sorted(g) for g in groups.values())
assert components_ans(5, [[0, 4], [4, 2], [1, 3]]) == [[0, 2, 4], [1, 3]]

# Bài 4
def best_arrangement(source, target, swaps):
    n = len(source); res = [None] * n
    for comp in components_ans(n, swaps):
        pool = Counter(source[i] for i in comp)
        rest = []
        for i in comp:
            if pool[target[i]] > 0:
                res[i] = target[i]; pool[target[i]] -= 1
            else:
                rest.append(i)
        leftovers = list(pool.elements())
        for i, v in zip(rest, leftovers):
            res[i] = v
    return res

random.seed(71)
for _ in range(400):
    n = random.randint(1, 6)
    src = [random.randint(1, 3) for _ in range(n)]; tgt = [random.randint(1, 3) for _ in range(n)]
    sw = [random.sample(range(n), 2) for _ in range(random.randint(0, 5))] if n > 1 else []
    arr = best_arrangement(src, tgt, sw)
    assert tuple(arr) in reachable_perms(src, sw)                       # đạt được thật
    assert hamming(arr, tgt) == min_hamming_bfs(src, tgt, sw)            # và tối ưu
print("Bài 1-4: OK")

Bài 1-4: OK


In [25]:
# Bài 5 — số phép đổi tối thiểu để đạt Hamming tối thiểu (BFS theo tầng)
def min_swaps_to_optimal(source, target, swaps):
    best = Solution().minimumHammingDistance(source, target, swaps)
    start = tuple(source); dist = {start: 0}; q = deque([start])
    while q:
        u = q.popleft()
        if hamming(u, target) == best:
            return best, dist[u]
        for a, b in swaps:
            v = list(u); v[a], v[b] = v[b], v[a]; v = tuple(v)
            if v not in dist:
                dist[v] = dist[u] + 1; q.append(v)

for (src, tgt, sw), _ in examples:
    print(src, tgt, sw, "-> (Hamming tối thiểu, số phép đổi ít nhất) =", min_swaps_to_optimal(src, tgt, sw))

[1, 2, 3, 4] [2, 1, 4, 5] [[0, 1], [2, 3]] -> (Hamming tối thiểu, số phép đổi ít nhất) = (1, 2)
[1, 2, 3, 4] [1, 3, 2, 4] [] -> (Hamming tối thiểu, số phép đổi ít nhất) = (2, 0)
[5, 1, 2, 4, 3] [1, 5, 4, 2, 3] [[0, 4], [4, 2], [1, 3], [1, 4]] -> (Hamming tối thiểu, số phép đổi ít nhất) = (0, 6)


## 20. Template nhớ nhanh + Checklist

```python
# Đổi chỗ tùy ý theo cặp  ->  DSU gom nhóm  ->  ghép đa tập trong từng nhóm
parent = list(range(n))
def find(x):
    root = x
    while parent[root] != root: root = parent[root]
    while parent[x] != root: parent[x], x = root, parent[x]
    return root
for a, b in allowedSwaps:
    parent[find(a)] = find(b)
cnt = defaultdict(Counter)
for i, x in enumerate(source): cnt[find(i)][x] += 1
ans = 0
for i, y in enumerate(target):
    c = cnt[find(i)]
    if c[y]: c[y] -= 1
    else: ans += 1
return ans
```

**Checklist trước khi submit:**
- [ ] Gom theo **thành phần liên thông**, không áp từng phép đổi đơn lẻ.
- [ ] Đếm **đa tập** (`Counter`), không phải `set`.
- [ ] Luôn dùng `find(i)` làm khóa nhóm.
- [ ] Không đệ quy sâu (`n = 10⁵`).
- [ ] `allowedSwaps` rỗng ⇒ đáp án là Hamming ban đầu.